In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git"
STUDENT_ROLE = "IT24100858 - Wijesinghe D.T.D"
STUDENT_NAME = input("Enter your real GitHub display name: ").strip()
STUDENT_EMAIL = input("Enter your verified GitHub email: ").strip()
CLONE_DIR = Path("/content/IT3091---Machine-Learning-Assignment")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
os.chdir(CLONE_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run(["git", "config", "user.name", STUDENT_NAME], check=True)
subprocess.run(["git", "config", "user.email", STUDENT_EMAIL], check=True)
print("Ready for", STUDENT_ROLE)
print("Authenticate your own GitHub account before running the final cell.")

# Production data preparation

This notebook reads the production workbook, removes unusable records, calculates yield, and saves the cleaned production table for the next member.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
raw_path = ROOT / 'data' / 'raw' / 'researchData.xlsx'
production = pd.read_excel(raw_path)
production['Year'] = pd.to_numeric(production['Year'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
production['Extent'] = pd.to_numeric(production['Extent'], errors='coerce')
production['Production'] = pd.to_numeric(production['Production'], errors='coerce')
production = production[production['Season'].isin(['Yala', 'Maha'])]
production = production[production['Year'].between(2012, 2023)]
production = production[production['District'].ne('National Total')]
production = production[(production['Extent'] > 0) & production['Production'].notna()].copy()
production['Yield_t_ha'] = production['Production'] / production['Extent']
district_map = {'Hanbantota': 'Hambantota', 'Kaluthara': 'Kalutara', 'Monaragala': 'Moneragala', 'Mullativu': 'Mullaitivu'}
production['District'] = production['District'].replace(district_map)
output = ROOT / 'data' / 'preprocessed'
output.mkdir(parents=True, exist_ok=True)
production.to_csv(output / 'production_clean.csv', index=False)
print('Raw rows:', len(pd.read_excel(raw_path)))
print('Clean production rows:', len(production))
print('Saved:', output / 'production_clean.csv')

The output keeps district, crop, season, year, cultivated extent, production, and the calculated yield. National-total rows are excluded from the modelling table.

In [ ]:
import subprocess

print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout)
subprocess.run(["git", "add", "notebooks/01_production_data_wijesinghe.ipynb"], check=True)
subprocess.run(["git", "add", "data/preprocessed/production_clean.csv"], check=True)
subprocess.run(["git", "commit", "-m", "Prepare cleaned production data"], check=False)
subprocess.run(["git", "push", "origin", "main"], check=True)
print("Contribution pushed to main.")